# 리포트 44 — 상시이면서 내용을 미리 아는 신호는 표준마다 하나씩 있다

> ### 한 일
> **WiFi · LTE · 5G NR 세 표준의 자원격자를 규격서대로 세우고, 패시브가 상관에 걸 수 있는 상시 기준신호를 표준마다 하나씩 골라 제원을 격자에서 직접 쟀다.**

### 결과
1. 두 조건(내용을 미리 안다 · 아무 셀이나 늘 켠다)을 함께 만족하는 신호는 표준마다 하나다 — WiFi VHT-LTF($B_{ref}$ 76.56 MHz [^1]) · LTE CRS(17.98 MHz [^2]) · 5G SSB(7.20 MHz [^3]).
2. 거리 눈금 $\Delta R_b = c/B_{ref}$ 는 3.9 [^4] · 16.7 [^5] · 41.6 m [^6] 로 세 표준이 한 자릿수 배 이상 갈린다.
3. $B_{ref}$ 는 기준신호가 차지한 부반송파의 양끝 span 이라 안쪽 널 톤을 포함한다 — WiFi 는 span 76.562 MHz [^1] 가 채널 점유대역 75.625 MHz [^7] 보다 넓다.
4. 프레임 안에서 기준신호가 실제로 차지하는 몫은 9.13% [^8] (WiFi) · 3.22% [^9] (LTE) · 1.45% [^10] (5G) 다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 자원격자 | `TS 36.211`(CRS) · `TS 38.211`(SSB) · `IEEE 802.11ac`(VHT-LTF) 를 읽어 세웠다 — `src/waveforms.py:258`(WiFi) · `:313`(LTE) · `:370`(5G) |
| 제원 측정 | 선언값을 옮겨 적지 않고 **생성한 격자에서 직접 쟀다** — $B_{ref}$ 는 `src/waveforms.py:237`, $\Delta R_b$ 는 `:144` |
| 거리 규약 | 바이스태틱 거리합 $R_b = R_1 + R_2 - L$ 이라 분해능은 $c/B_{ref}$ 다 — 모노스태틱 교과서 값의 두 배다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python src/viz_report2.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json`, `outputs/report03_illuminators.json` |
| 소요 | ① 3412 s [^11] (대부분 같은 스크립트의 RCS 스윕) · ② CPU 20초 안쪽 |

---

## 패시브가 상관을 걸 수 있는 신호는 어떤 것인가

패시브 수신기는 남이 쏘는 신호를 빌려 쓴다. 그 신호에 상관을 걸려면 두 조건이 **동시에** 서야 한다.

**① 내용을 미리 안다.** 데이터는 매 순간 바뀌므로 규격이 고정한 기준신호가 그 자리를 맡는다.

**② 아무 셀이나 늘 켠다.** 상시 신호라야 표적이 지나가는 그 순간에도 공중에 있다.

두 조건을 다 만족하는 신호는 표준마다 **하나씩**이다.

## 격자에서 잰 제원

| 표준 | 상시 기준신호 | 반송파 | 채널 점유대역 | $B_{ref}$ | $\Delta R_b=c/B_{ref}$ |
|---|---|---|---|---|---|
| WiFi 802.11ac | VHT-LTF | 5.21 GHz [^12] | 75.6 MHz [^7] | 76.6 MHz [^1] | 3.9 m [^4] |
| LTE Rel-9 | CRS | 1.843 GHz [^13] | 18.0 MHz [^14] | 18.0 MHz [^2] | 16.7 m [^5] |
| 5G NR Rel-16 | SSB | 3.50 GHz [^15] | 98.3 MHz [^16] | 7.2 MHz [^3] | 41.6 m [^6] |

## $B_{ref}$ 는 span 이다

$B_{ref}$ 는 기준신호가 차지한 부반송파의 **양끝 span** 이다(`src/waveforms.py:237`). 안쪽 널 톤이 그 안에 들어오므로 WiFi 는 span 이 점유대역보다 넓게 나온다.

이 정의가 거리 눈금을 정한다. 채널 대역이 아니라 **상관에 쓰는 대역**이 분해능을 만들기 때문이다 — 5G 채널은 98.3 MHz [^16] 인데 상시 SSB 체제의 거리 눈금은 41.6 m [^6] 다. 채널 대역이 다 열린 체제의 값 3.05 m [^17] 는 [편 46 «여섯 항목은 닫힌형이고»](46_cost-ledger.ipynb) 가 낙관적 상한으로 함께 싣는다.

규약의 정확한 형태는 [편 47 «바이스태틱 거리 분해능은 c/B, 잡음대역은 √(B/fs) 로 고정한다»](47_range-convention.ipynb) 가 든다.

![report03_f1_grid](../outputs/figures/report03_f1_grid.png)

**그림 1.** 유휴 셀이 실제로 켜는 칸은 어디이고, 그중 패시브가 상관에 쓰는 것은 무엇인가?

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| `src/waveforms.py:112` 의 `PILOT_RATE_HZ` 를 트래픽 시나리오 파라미터로 올린다 | WiFi PRF 가 유휴 AP ~ 혼잡 AP 범위로 확정되고 이 편의 제원표가 시나리오별로 선다 | `src/waveforms.py:112` |
| X410 으로 실제 셀을 캡처해 격자 좌표를 대조한다 | CRS · SSB · VHT-LTF 의 자원요소 좌표가 실측으로 확정된다 | [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 17개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.ref_bw_mhz` | 76.56 |
| [^2] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.ref_bw_mhz` | 17.98 |
| [^3] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.ref_bw_mhz` | 7.2 |
| [^4] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.dR_m` | 3.916 |
| [^5] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.dR_m` | 16.67 |
| [^6] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.dR_m` | 41.64 |
| [^7] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.chan_bw_mhz` | 75.62 |
| [^8] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.occ_pct` | 9.135 |
| [^9] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.occ_pct` | 3.223 |
| [^10] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.occ_pct` | 1.447 |
| [^11] | `outputs/report2_waveform_rcs.json` | `meta.runtime_s` | 3412 |
| [^12] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.carrier_ghz` | 5.21 |
| [^13] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.carrier_ghz` | 1.843 |
| [^14] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.chan_bw_mhz` | 18 |
| [^15] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.carrier_ghz` | 3.5 |
| [^16] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.chan_bw_mhz` | 98.28 |
| [^17] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.chan_dR_m` | 3.05 |